# Synthetic Biology Pipeline - Interactive Analysis

This notebook demonstrates how to use the metabolic pathway design pipeline interactively.

## Table of Contents
1. [Setup](#Setup)
2. [Run Full Pipeline](#Run-Full-Pipeline)
3. [Analyze Results](#Analyze-Results)
4. [Compare Organisms](#Compare-Organisms)
5. [Visualization](#Visualization)

In [ ]:
# Setup imports
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import pipeline modules
from main_pipeline_runner import PipelineRunner
from pipeline_config import PipelineConfig

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Run Full Pipeline

In [ ]:
# Initialize runner
runner = PipelineRunner()

# Run pipeline for E. coli + Lycopene
results = runner.run_full_pipeline(
    organism="ecoli",
    target_molecule="lycopene",
    dbtl_cycles=3,
    output_dir="./pipeline_output"
)

print(f"Pipeline ID: {results['pipeline_id']}")
print(f"Status: {results.get('overall_status', 'UNKNOWN')}")

## Analyze Results

In [ ]:
# Extract key metrics
stage4 = results.get('stage_4', {})
stage5 = results.get('stage_5', {})

fermentation = stage4.get('fermentation_simulation', {})
scaleup = stage5.get('scale_up_prediction', {})
regulatory = stage5.get('regulatory_assessment', {})

print("=== Fermentation Results ===")
print(f"Final Titer: {fermentation.get('final_titer_g_per_l', 'N/A')} g/L")
print(f"Yield: {fermentation.get('final_yield_g_per_g', 'N/A')} g/g")
print(f"Productivity: {fermentation.get('final_productivity_g_per_l_per_h', 'N/A')} g/L/h")

print("\n=== Scale-up Prediction ===")
print(f"Production Titer: {scaleup.get('production_titer_g_per_l', 'N/A')} g/L")
print(f"Yield Loss: {scaleup.get('yield_loss_percent', 'N/A')}%")

print("\n=== Regulatory Assessment ===")
print(f"Biosafety Level: {regulatory.get('biosafety_level', 'N/A')}")
print(f"GRAS Status: {regulatory.get('gras_status', 'N/A')}")
print(f"Compliance Score: {regulatory.get('compliance_score', 'N/A')}/100")

## Compare Organisms

In [ ]:
# Compare multiple organisms
organisms = ['ecoli', 'scerevisiae', 'bsubtilis', 'cglutamicum']
results_dict = {}

for org in organisms:
    print(f"Running pipeline for {org}...")
    try:
        res = runner.run_full_pipeline(
            organism=org,
            target_molecule="lycopene",
            dbtl_cycles=2,  # Fewer cycles for faster comparison
            output_dir=f"./pipeline_output_{org}"
        )
        results_dict[org] = res
    except Exception as e:
        print(f"Error with {org}: {e}")
        results_dict[org] = None

# Create comparison DataFrame
comparison_data = []
for org, res in results_dict.items():
    if res:
        fermen = res.get('stage_4', {}).get('fermentation_simulation', {})
        comparison_data.append({
            'Organism': org,
            'Titer (g/L)': fermen.get('final_titer_g_per_l', 0),
            'Yield (g/g)': fermen.get('final_yield_g_per_g', 0),
            'Productivity (g/L/h)': fermen.get('final_productivity_g_per_l_per_h', 0)
        })

df_compare = pd.DataFrame(comparison_data)
print("\nComparison Table:")
print(df_compare.to_string(index=False))

## Visualization

In [ ]:
# Plot comparison
if len(df_compare) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    df_compare.plot.bar(x='Organism', y='Titer (g/L)', ax=axes[0], legend=False)
    axes[0].set_title('Final Titer by Organism')
    axes[0].set_ylabel('g/L')
    
    df_compare.plot.bar(x='Organism', y='Yield (g/g)', ax=axes[1], legend=False, color='orange')
    axes[1].set_title('Yield by Organism')
    axes[1].set_ylabel('g/g')
    
    df_compare.plot.bar(x='Organism', y='Productivity (g/L/h)', ax=axes[2], legend=False, color='green')
    axes[2].set_title('Productivity by Organism')
    axes[2].set_ylabel('g/L/h')
    
    plt.tight_layout()
    plt.savefig('./pipeline_output/organism_comparison.png', dpi=150)
    plt.show()

In [ ]:
# Load and display fermentation time series if available
output_dir = Path('./pipeline_output')
report_file = output_dir / 'final_report.json'

if report_file.exists():
    with open(report_file) as f:
        full_report = json.load(f)
    
    # Display DBTL cycle improvement
    dbtl_cycles = full_report.get('stage_4', {}).get('dbtl_cycles', [])
    if dbtl_cycles:
        cycle_data = pd.DataFrame(dbtl_cycles)
        
        plt.figure(figsize=(10, 5))
        plt.plot(cycle_data['cycle_number'], cycle_data['best_titer_g_per_l'], marker='o', linewidth=2)
        plt.xlabel('DBTL Cycle')
        plt.ylabel('Best Titer (g/L)')
        plt.title('Improvement Across DBTL Cycles')
        plt.grid(True, alpha=0.3)
        plt.savefig('./pipeline_output/dbtl_improvement.png', dpi=150)
        plt.show()
        
        print("\nDBTL Cycle Summary:")
        print(cycle_data.to_string(index=False))